# Eukaryotic toehold switch — two-input AND

Manual test harness for `engine.gates.toehold.ToeholdAndGate`, built against a
**eukaryotic host** (`Host.HUMAN`/`Host.YEAST`, Kozak-cap-dependent-scanning
track — see `ToeholdGate`'s own docstring on why `host` is a constructor
parameter rather than a subclass). It inherits every method from `ToeholdGate`
except `generate_designs`, and sets `max_inputs = 2`. Scientific methods print
`pending Step 5` until the bodies land.

This notebook is `toehold_and.ipynb`'s sibling, not a replacement — that one
exercises the same stub against `Host.ECOLI` (steric RBS occlusion, serial
hairpins). Kept separate rather than parameterised over host in one notebook
because, once Step 5 lands, the two tracks are expected to diverge in real
construction (Kozak/scanning-blockage vs. RBS/steric-occlusion — see
`docs/modalities.md` and `ToeholdGate`'s own single-input eukaryotic work,
`toehold/eukaryotic-usage-example.ipynb`, for how much that single-input case
already had to diverge), not just in which fixed motif gets substituted in.

## The mechanism

An AND toehold opens only when **both** triggers are present. The
single-input eukaryotic case (`EukaryoticToeholdGate`, already built — see
`toehold/eukaryotic-usage-example.ipynb`) blocks a scanning 40S ribosome with a
closed hairpin sitting *upstream* of Kozak+AUG, not by sequestering Kozak
itself the way the prokaryotic RBS-in-loop layout sequesters the RBS. Extending
that to two inputs raises questions the prokaryotic serial-stem shape doesn't
have to answer, none of them resolved here — this notebook watches for them
once Step 5 lands, it doesn't decide them:

* **Where do two hairpins sit relative to one Kozak?** The prokaryotic AND
  puts the start codon inside the *inner* of two nested hairpins, with an RBS
  free in each hairpin's own loop. A `"trailing"`-style eukaryotic construct has
  no RBS-in-loop to nest a second one inside — the open question is whether the
  outer hairpin blocks scanning *before* it ever reaches an inner hairpin+Kozak,
  or whether Kozak itself gets sequestered by the inner stem the way the
  prokaryotic AUG does today.
* **Order still matters** — A-outer/B-inner is a different construct from the
  reverse, same as the prokaryotic case.
* **The intermediate (one-trigger) state is real** — if a scanning ribosome can
  already reach Kozak+AUG with only the first trigger present, the gate is an OR
  wearing an AND's shape. Evaluate the single-trigger states explicitly once
  `evaluate_design` exists for this class.
* **Kozak self-binding risk, doubled.** The single-input case already tracks
  whether a trigger-derived toehold happens to carry Kozak's reverse complement
  (`_kozak_rc_in_toehold`, and its worst-case `predicted_leakage` penalty in
  `evaluate_design` — eukaryotic-only, gated on `self.host.track`). An AND gate
  has two trigger-derived toeholds and one Kozak copy; whichever construction
  Step 5 picks needs the same check run against *both*, not just the one
  nearest Kozak in sequence.

## Setup

In [ ]:
# Put the shared _fixtures.py on the path. It lives in the notebooks/ root, one
# level up from this gate's folder; search upward so the notebook works wherever
# Jupyter is launched. fx.bootstrap() then adds <repo>/src so `import engine...`
# resolves. No Django, no worker, no pipeline.
import sys, pathlib

for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / '_fixtures.py').exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break

import _fixtures as fx
fx.bootstrap()

## Build the gate

In [ ]:
host = fx.Host.HUMAN  # HUMAN | YEAST both take the eukaryotic (Kozak) track
gate = fx.toehold_and(host=host)
fx.describe_gate(gate)

## Inputs

A `TriggerSet` (the circuit's inputs) and a `Constraints` (the researcher's
limits). Both are made up here — override any field via keyword.

In [ ]:
triggers = fx.sample_trigger_set(n_activators=2)   # two activators; may share a gene
constraints = fx.sample_constraints(max_switch_length=240)

for t in triggers.activators:
    print(f'{t.trigger_id}  {t.symbol:6}  {t.sequence}  {t.length} nt')
print('arity:', triggers.arity, '| logic:', triggers.logic_type)

## `required_tools()` — implemented

In [ ]:
fx.attempt('required_tools', gate.required_tools)

## `is_compatible()` *(Step 5)*

Try it with a one-activator set too — a two-input family should reject arity 1
with a message written for the researcher.

In [ ]:
fx.attempt('is_compatible (2 activators)', lambda: gate.is_compatible(triggers, constraints))
one = fx.sample_trigger_set(n_activators=1)
fx.attempt('is_compatible (1 activator)', lambda: gate.is_compatible(one, constraints))

## `generate_designs()` *(Step 5)*

Expect both trigger orders to be generated, and the single-trigger states
evaluated explicitly. For this host, also expect whichever layout(s) Step 5
settles on for a eukaryotic AND to be swept the way the single-input gate
sweeps `kozak_layouts` — not silently fixed to one shape.

In [ ]:
designs = fx.attempt(
    'generate_designs',
    lambda: list(gate.generate_designs(triggers, constraints)),
)

## `evaluate_design()` *(Step 5)*

In [ ]:
target = designs[0] if designs else fx.sample_design(gate, triggers)
fx.attempt('evaluate_design', lambda: gate.evaluate_design(target))

## `emit_sequence()` and `describe()` — output helpers

These two are implemented today. `generate_designs()` is not, so
`fx.sample_design(...)` hands us a plausible `GateDesign` to call them on.

In [ ]:
design = fx.sample_design(gate, triggers)
print('design_id  ', design.design_id)
print('gate_kind  ', design.gate_kind)
print('length     ', design.length, 'nt')
print('emit_sequence:', gate.emit_sequence(design))
print('describe     :', gate.describe(design))

## Switching in real folding

Everything above runs against `fx.StubFoldEngine` — deterministic, fake, no
ViennaRNA. For genuine structure predictions, pass `real_fold=True` when you build
the gate (needs `import RNA` to work in this environment):

```python
gate = fx.toehold_and(host=host, real_fold=True)
folder = fx.fold_engine(real=True)
folder.mfe('GGGAAACCCUUUGGGAAACCC')   # -> FoldResult(structure, energy)
```

## Fusing two ranked single-input designs into an AND candidate

`ToeholdAndGate.generate_designs` is still `NotImplementedError("Step 5")`, so nothing
below asks the gate to build anything. This section takes two **already-ranked
single-input eukaryotic toeholds** — one CSV per target — splices each A x B pair into
one serial construct, and measures it. It is a notebook-level construction; when Step 5
lands, `generate_designs` owns this and this section becomes the thing to check against.

**The inputs** are NucSyn runner exports (`<target>/<profile>/results_ranked.csv`,
topology `euk_open_5p_kozak_after_stem`), already sorted best-first by
`postgen_global_rank`. Three columns carry the work:

| column | what it gives |
|---|---|
| `switch_strand` | the full designed construct |
| `trigger_seq` | the trigger that opens it |
| `switch_domain_breakdown_json` | named, 0-indexed, start-inclusive/end-exclusive domain bounds over `switch_strand` |

**Why the domain breakdown and not string matching.** `kozak_seq` in these exports is
the degenerate IUPAC *pattern* (`GCMAMMAUG`, M = A or C), not the sequence that actually
landed in the construct, so `switch_strand.endswith(kozak_seq)` is always false. The
construct also does not end at the Kozak — 30 nt of reporter CDS (`exp_gene`) follow it.
The breakdown names every domain and its exact bounds, in the same 0-indexed half-open
convention this repo uses (`CLAUDE.md` §6), so the cut points are read rather than
guessed.

**The construction.** A design here reads

```
prefix | toehold | stem_up | loop | stem_down | pre_kozak | kozak | ext_kozak | opt_exp_gene | exp_gene
```

A serial AND keeps exactly one of each fixed end:

* **A** keeps everything **before its `kozak` domain starts** — so A's prefix stays at the
  5' end and A's hairpin comes first, while A's Kozak, start codon and reporter head are
  dropped.
* **B** keeps everything **after its `prefix` domain ends** — so B's hairpin comes second
  and B's Kozak+AUG+reporter lands at the 3' end, where a scanning ribosome reaches it
  last.

Both hairpins therefore sit in series between one prefix and one Kozak — the same shape
`process_trigger_p_and` builds in the team's own `CERNAL_FUNCTIONS.py`.

**What gets measured**, for every A x B pair, as raw kcal/mol:

| state | folded as | what it says |
|---|---|---|
| OFF | fused switch alone | both hairpins closed, nothing bound |
| +A | `fused&triggerA` | trigger A only — the intermediate state |
| +B | `fused&triggerB` | trigger B only — the other intermediate state |
| +A+B | `fused&triggerA&triggerB` | the state the gate is supposed to need |

Every multi-molecule state goes through ViennaRNA's `&` syntax (`FoldEngine.mfe` and
`.partition` both accept it, and ViennaRNA 2.7 folds three strands), never string
concatenation — concatenating covalently joins molecules that are not joined and returns
a plausible number that means nothing (`CLAUDE.md` §6).

**Ranking is a notebook stand-in, not `engine.scoring`** (`CLAUDE.md` §3). The sort key is
one named constant in the run cell, and the ΔG sign convention is spelled out there —
"highest" is ambiguous for a quantity that is negative when it is good.

In [ ]:
# --- inputs -------------------------------------------------------------------
# NucSyn runner exports, already sorted best-first by postgen_global_rank. This section
# preserves that order and never re-ranks within a file.
CSV_A = "/Users/zeev/Projects/TOEHOLDS/areg/basic_switch/results_ranked.csv"
CSV_B = "/Users/zeev/Projects/TOEHOLDS/sele/basic_switch/results_ranked.csv"

# A is the 5' half (keeps its prefix + hairpin), B is the 3' half (keeps its hairpin +
# Kozak + reporter head). Swapping these two paths builds the other trigger order, which
# is a different construct -- see this notebook's mechanism section.
COLUMNS = {
    "switch": "switch_strand",
    "trigger": "trigger_seq",
    "domains": "switch_domain_breakdown_json",
    "rank": "postgen_global_rank",
}

# The producing platform's own per-design numbers, carried through for context and
# never recomputed here (recomputing with a slightly different model would produce two
# numbers for one quantity). sim_score is its overall objective and is **lower-is-better**
# -- it is what postgen_global_rank is built from. The rest are the components the
# length trade-off actually shows up in: structure fidelity, toehold openness, and how
# much the trigger folds on itself.
CARRY_METRICS = (
    "sim_score",
    "mcc",
    "switch_toehold_unpaired_prob_mean",
    "energy_trigger_mfe",
)

# Where each half is cut, by domain name rather than by length.
A_CUT_BEFORE = "kozak"  # A keeps everything before its kozak domain starts
B_KEEP_AFTER = "prefix"  # B keeps everything after its prefix domain ends

# Folding is ~4 folds per pair (OFF, +A, +B, +A+B) and the grid is TOP_A x TOP_B, so
# start small and raise it once a run looks right. 1200 x 1200 is not a thing to launch
# by accident.
TOP_A = 10
TOP_B = 10

TEMPERATURE_C = 37.0  # recorded on every energy below; see FoldEngine's docstring

In [ ]:
import csv
import json
from pathlib import Path


def _maybe_float(value):
    """``float(value)``, or ``None`` when the cell is blank or not a number.

    Deliberately not ``float(value or 0)``: a blank cell is a measurement that is not
    there, and 0.0 is a real value that would go on to sort as though it had been
    measured (``CLAUDE.md`` §3).
    """
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def load_ranked_designs(path, columns=COLUMNS, limit=None):
    """Rows of a NucSyn ``results_ranked.csv``, kept in file order (already ranked).

    Returns ``(rows, problems)``. A row missing a column, carrying a non-RNA sequence, or
    whose domain breakdown does not tile its own switch exactly is collected in
    ``problems`` with its line number and reason — never dropped silently or patched to a
    default.

    Sequences are normalised with ``engine.sequences.to_rna`` (re-exported as ``fx.sq``)
    here at the edge: a CSV is exactly where a DNA alphabet sneaks in, and ViennaRNA
    reads a T as an unknown base rather than failing (``CLAUDE.md`` §6).
    """
    rows, problems = [], []
    with Path(path).open(newline="") as handle:
        for line_no, raw in enumerate(csv.DictReader(handle), start=2):  # 2 = first data line
            missing = [column for column in columns.values() if not (raw.get(column) or "").strip()]
            if missing:
                problems.append((line_no, f"missing or empty: {', '.join(missing)}"))
                continue

            switch = fx.sq.to_rna(raw[columns["switch"]])
            trigger = fx.sq.to_rna(raw[columns["trigger"]])
            not_rna = [
                name
                for name, value in (("switch", switch), ("trigger", trigger))
                if not fx.sq.is_valid_rna(value)
            ]
            if not_rna:
                problems.append((line_no, f"not RNA after to_rna(): {', '.join(not_rna)}"))
                continue

            # 0-indexed, start-inclusive/end-exclusive, same convention as the engine.
            domains = {
                domain["name"]: (domain["start"], domain["end"])
                for domain in json.loads(raw[columns["domains"]])
            }
            # The breakdown is the only thing locating the cut points, so a breakdown that
            # does not tile the switch exactly would put those cuts somewhere arbitrary.
            covered = sorted(domains.values())
            if not covered or covered[0][0] != 0 or covered[-1][1] != len(switch):
                problems.append((line_no, "domain breakdown does not span the switch"))
                continue

            rows.append(
                {
                    "switch": switch,
                    "trigger": trigger,
                    "domains": domains,
                    "rank": raw[columns["rank"]],
                    "metrics": {name: _maybe_float(raw.get(name)) for name in CARRY_METRICS},
                    "line_no": line_no,
                }
            )
            if limit is not None and len(rows) >= limit:
                break
    return rows, problems

In [ ]:
def fuse(design_a, design_b, cut_before=A_CUT_BEFORE, keep_after=B_KEEP_AFTER):
    """One AND candidate: A up to its Kozak, then B after its prefix.

    Returns ``(fused, problem)``. ``problem`` is non-None when either design lacks the
    domain its cut is defined by — slicing a guessed offset instead would still return a
    sequence, and it would fold into a plausible-looking wrong answer nothing downstream
    could catch.

    Note for other topologies: cutting A at ``kozak`` leaves A's ``pre_kozak`` arm in
    place while dropping the ``post_kozak`` it was designed to pair with. Both are
    zero-length in ``euk_open_5p_kozak_after_stem``, so nothing is orphaned here; a
    topology that uses them needs this cut reconsidered, not just re-pointed.
    """
    if cut_before not in design_a["domains"]:
        return None, f"A line {design_a['line_no']}: no {cut_before!r} domain to cut before"
    if keep_after not in design_b["domains"]:
        return None, f"B line {design_b['line_no']}: no {keep_after!r} domain to keep after"

    body_a = design_a["switch"][: design_a["domains"][cut_before][0]]
    body_b = design_b["switch"][design_b["domains"][keep_after][1] :]
    return body_a + body_b, None

In [ ]:
def measure(fused, trigger_a, trigger_b, folder):
    """MFE and ensemble free energy for the four states, all raw kcal/mol.

    ``FoldEngine`` caches per sequence, so the shared instance built in the run cell
    below is the only one that should ever fold here — a second instance means a cold
    cache and, worse, a second chance to fold at a different temperature.
    """
    states = {
        "off": fused,  # no trigger: both hairpins closed
        "a": f"{fused}&{trigger_a}",  # '&' = a true multi-strand complex, not a fusion
        "b": f"{fused}&{trigger_b}",
        "ab": f"{fused}&{trigger_a}&{trigger_b}",
    }

    out = {}
    for name, strands in states.items():
        out[f"mfe_{name}"] = folder.mfe(strands).energy
        out[f"ee_{name}"] = folder.partition(strands)

    # dG of a state = that state minus the trigger-free OFF state, so a negative number is
    # the stabilisation that adding trigger(s) bought.
    for name in ("a", "b", "ab"):
        out[f"d_mfe_{name}"] = out[f"mfe_{name}"] - out["mfe_off"]
        out[f"d_ee_{name}"] = out[f"ee_{name}"] - out["ee_off"]

    # Reported, never ranked on: how much the second trigger buys beyond whichever single
    # trigger already did most of the work. Near zero means one trigger alone already
    # opens the construct -- an OR wearing an AND's shape, which is the specific failure
    # this architecture has to be checked for (see this notebook's mechanism section).
    out["and_margin_ee"] = min(out["d_ee_a"], out["d_ee_b"]) - out["d_ee_ab"]
    return out

In [ ]:
# dG with both triggers, from the ensemble rather than the single MFE structure.
RANK_KEY = "d_ee_ab"
# True = most negative first, i.e. the largest energy gain from having both triggers.
# Flip to False to read "highest" literally as least-negative instead.
RANK_STRONGEST_FIRST = True

designs_a, problems_a = load_ranked_designs(CSV_A, limit=TOP_A)
designs_b, problems_b = load_ranked_designs(CSV_B, limit=TOP_B)
print(f"A: {len(designs_a)} design(s) from {CSV_A}")
print(f"B: {len(designs_b)} design(s) from {CSV_B}")
for label, problems in (("A", problems_a), ("B", problems_b)):
    for line_no, reason in problems:
        print(f"  skipped {label} line {line_no}: {reason}")

# One FoldEngine for the whole grid -- see measure()'s docstring.
folder = fx.fold_engine(real=True, temperature=TEMPERATURE_C)

results, skipped = [], []
for design_a in designs_a:
    for design_b in designs_b:
        fused, problem = fuse(design_a, design_b)
        if problem is not None:
            skipped.append(problem)
            continue
        row = measure(fused, design_a["trigger"], design_b["trigger"], folder)
        row.update(
            a_rank=design_a["rank"],
            b_rank=design_b["rank"],
            a_metrics=design_a["metrics"],
            b_metrics=design_b["metrics"],
            fused=fused,
            length=len(fused),
            trigger_a=design_a["trigger"],
            trigger_b=design_b["trigger"],
        )
        results.append(row)

for problem in skipped:
    print(f"  skipped pair: {problem}")

results.sort(key=lambda row: row[RANK_KEY], reverse=not RANK_STRONGEST_FIRST)

print(f"\n{len(results)} fused candidate(s), best first by {RANK_KEY}:\n")
# Trigger lengths are printed because raw dG scales with how many base pairs a trigger
# can form: a longer trigger buys a more negative dG whether or not the switch is any
# better, so a ranking on dG alone quietly sorts by trigger length. Shown, not corrected.
def _fmt(value, width=7, places=2):
    """A measurement that is not there prints as '-', not as a number."""
    return f"{'-':>{width}}" if value is None else f"{value:>{width}.{places}f}"


header = (
    f"{'#':>3}  {'A':>4} {'B':>4}  {'ntA':>4} {'ntB':>4}  {'nt':>4}  {'ee_off':>8}  "
    f"{'dG +A':>7}  {'dG +B':>7}  {'dG +A+B':>8}  {'AND margin':>10}  "
    f"{'simA':>7} {'simB':>7}"
)
print(header)
print("-" * len(header))
# simA/simB are the producer's own verdict on each half, lower is better. They pull
# against dG here: in these exports a longer trigger buys several kcal/mol of binding
# while scoring ~1 point worse on sim_score, folding on itself more, and matching its
# intended structure slightly less well. Both numbers are shown so that trade is a
# choice rather than an accident of whichever one got sorted on.
for position, row in enumerate(results[:20], start=1):
    print(
        f"{position:>3}  {row['a_rank']:>4} {row['b_rank']:>4}  "
        f"{len(row['trigger_a']):>4} {len(row['trigger_b']):>4}  {row['length']:>4}  "
        f"{row['ee_off']:>8.1f}  {row['d_ee_a']:>7.1f}  {row['d_ee_b']:>7.1f}  "
        f"{row['d_ee_ab']:>8.1f}  {row['and_margin_ee']:>10.1f}  "
        f"{_fmt(row['a_metrics']['sim_score'])} {_fmt(row['b_metrics']['sim_score'])}"
    )

if results:
    best = results[0]
    print(f"\nbest: A rank {best['a_rank']} + B rank {best['b_rank']}, {best['length']} nt")
    print(f"  trigger A     {best['trigger_a']}")
    print(f"  trigger B     {best['trigger_b']}")
    print(f"  fused switch  {best['fused']}")